<a href="https://colab.research.google.com/github/wadaglizz/DSA4060XA-Recommender-Project/blob/main/DSA4060XA_Dataset_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests

url = "https://mcauleylab.ucsd.edu/public_datasets/data/steam/australian_users_items.json.gz"

response = requests.get(url, timeout=180)
response.raise_for_status()

with open("/content/australian_users_items.json.gz", "wb") as f:
    f.write(response.content)

print("Dataset downloaded successfully!")
print("File size:", round(len(response.content) / (1024**2), 2), "MB")


Dataset downloaded successfully!
File size: 70.61 MB


In [2]:
import gzip
import ast
import random
import pandas as pd

random.seed(42)

file_path = "/content/australian_users_items.json.gz"

sample = []
total_interactions = 0
sample_size = 5000

with gzip.open(file_path, "rt", encoding="utf-8") as file:

    for line in file:

        user = ast.literal_eval(line)

        user_id = user.get("user_id")

        for game in user.get("items", []):

            interaction = {
                "user_id": user_id,
                "game_id": game.get("item_id"),
                "game_name": game.get("item_name"),
                "playtime_forever": game.get("playtime_forever", 0),
                "playtime_2weeks": game.get("playtime_2weeks", 0)
            }

            total_interactions += 1

            if len(sample) < sample_size:
                sample.append(interaction)

            else:
                j = random.randint(0, total_interactions - 1)

                if j < sample_size:
                    sample[j] = interaction

df = pd.DataFrame(sample)

print("Total original interactions:", total_interactions)
print("Sample size:", len(df))
display(df.head())

Total original interactions: 5153209
Sample size: 5000


,user_id,game_id,game_name,playtime_forever,playtime_2weeks
0,76561197998875095,298110,Far Cry 4,1592,431
1,76561197999375514,94590,Puzzle Agent 2,0,0
2,necrotos,461910,Forge of Gods (RPG),0,0
3,76561198084480904,39120,RIFT,666,0
4,Meyjax,218130,Dungeonland,0,0


In [3]:
print("Dataset shape:", df.shape)

print("\nColumn information:")
print(df.info())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate interactions:")
print(df.duplicated(
    subset=["user_id", "game_id"]
).sum())

print("\nUnique users:", df["user_id"].nunique())
print("Unique games:", df["game_id"].nunique())

print("\nPlaytime statistics:")
print(df["playtime_forever"].describe())

Dataset shape: (5000, 5)

Column information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   user_id           5000 non-null   object
 1   game_id           5000 non-null   object
 2   game_name         5000 non-null   object
 3   playtime_forever  5000 non-null   int64 
 4   playtime_2weeks   5000 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 195.4+ KB
None

Missing values:
user_id             0
game_id             0
game_name           0
playtime_forever    0
playtime_2weeks     0
dtype: int64

Duplicate interactions:
0

Unique users: 4464
Unique games: 1848

Playtime statistics:
count      5000.000000
mean       1026.495200
std        5497.427756
min           0.000000
25%           0.000000
50%          32.000000
75%         355.250000
max      132650.000000
Name: playtime_forever, dtype: float64


In [4]:
!pip -q install pyarrow

df.to_parquet(
    "/content/steam_gaming_5000.parquet",
    index=False
)

print("5,000 interactions saved successfully!")

5,000 interactions saved successfully!


In [5]:
# User and game interaction analysis

user_interactions = df.groupby("user_id").size()
game_interactions = df.groupby("game_id").size()

print("USER INTERACTION DISTRIBUTION")
print(user_interactions.describe())

print("\nUsers with only one interaction:")
print((user_interactions == 1).sum())

print("\nUsers with at least 2 interactions:")
print((user_interactions >= 2).sum())

print("\nUsers with at least 5 interactions:")
print((user_interactions >= 5).sum())

print("\nGAME INTERACTION DISTRIBUTION")
print(game_interactions.describe())

print("\nZero playtime interactions:")
print((df["playtime_forever"] == 0).sum())

print("\nPercentage with zero playtime:")
print(round((df["playtime_forever"] == 0).mean() * 100, 2), "%")

USER INTERACTION DISTRIBUTION
count    4464.000000
mean        1.120072
std         0.438879
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         9.000000
dtype: float64

Users with only one interaction:
4041

Users with at least 2 interactions:
423

Users with at least 5 interactions:
12

GAME INTERACTION DISTRIBUTION
count    1848.000000
mean        2.705628
std         3.873463
min         1.000000
25%         1.000000
50%         1.000000
75%         3.000000
max        56.000000
dtype: float64

Zero playtime interactions:
1803

Percentage with zero playtime:
36.06 %


In [6]:
import gzip
import ast
import random
import pandas as pd

random.seed(42)

file_path = "/content/australian_users_items.json.gz"

target_users = 500
games_per_user = 10

selected_users = []
eligible_users = 0

with gzip.open(file_path, "rt", encoding="utf-8") as file:

    for line in file:

        user = ast.literal_eval(line)

        user_id = user.get("user_id")
        items = user.get("items", [])

        # Remove duplicate game IDs within each user
        unique_items = {}

        for game in items:
            game_id = game.get("item_id")

            if game_id is not None:
                unique_items[game_id] = game

        items = list(unique_items.values())

        # Require at least 10 games per user
        if len(items) < games_per_user:
            continue

        eligible_users += 1

        # Reservoir sampling of eligible users
        if len(selected_users) < target_users:

            chosen = random.sample(items, games_per_user)

            selected_users.append(
                (user_id, chosen)
            )

        else:

            j = random.randrange(eligible_users)

            if j < target_users:

                chosen = random.sample(
                    items, games_per_user
                )

                selected_users[j] = (
                    user_id, chosen
                )

# Flatten selected user-game records
records = []

for user_id, games in selected_users:

    for game in games:

        records.append({
            "user_id": user_id,
            "game_id": game.get("item_id"),
            "game_name": game.get("item_name"),
            "playtime_forever": game.get(
                "playtime_forever", 0
            ),
            "playtime_2weeks": game.get(
                "playtime_2weeks", 0
            )
        })

balanced_df = pd.DataFrame(records)

print("Eligible users:", eligible_users)
print("Selected users:", len(selected_users))
print("Dataset shape:", balanced_df.shape)
print("Unique users:", balanced_df["user_id"].nunique())
print("Unique games:", balanced_df["game_id"].nunique())

Eligible users: 57892
Selected users: 500
Dataset shape: (5000, 5)
Unique users: 500
Unique games: 1348


In [7]:
print("Missing values:")
print(balanced_df.isnull().sum())

print("\nDuplicate user-game pairs:")
print(
    balanced_df.duplicated(
        subset=["user_id", "game_id"]
    ).sum()
)

print("\nInteractions per user:")
print(balanced_df.groupby("user_id").size().describe())

print("\nZero playtime:")
print(
    (balanced_df["playtime_forever"] == 0).sum()
)

balanced_df.to_parquet(
    "/content/steam_gaming_5000.parquet",
    index=False
)

print("\nBalanced dataset saved!")

Missing values:
user_id             0
game_id             0
game_name           0
playtime_forever    0
playtime_2weeks     0
dtype: int64

Duplicate user-game pairs:
0

Interactions per user:
count    500.0
mean      10.0
std        0.0
min       10.0
25%       10.0
50%       10.0
75%       10.0
max       10.0
dtype: float64

Zero playtime:
1572

Balanced dataset saved!


In [8]:
# Preserve the original random sample
df.to_parquet(
    "/content/steam_gaming_random_5000.parquet",
    index=False
)

# Save the balanced modelling sample
balanced_df.to_parquet(
    "/content/steam_gaming_5000.parquet",
    index=False
)

print("Both datasets saved successfully.")

Both datasets saved successfully.
